# 🎧 00 · 语音信号基础与特征提取

语音 = 空气振动 → 麦克风振动膜 → 电信号 → **ADC 数字化** → **分帧加窗** → **特征** → 模型。

本系列 5 讲：**00 信号与特征（本节）→ 01 GMM-HMM 传统识别 → 02 CTC/Attention 端到端 → 03 TTS 合成 → 04 面试八股**。

本节目标：把「一段 wav」变成「一行特征向量」，并理解每一步为什么这么做。全程 numpy 手写，无外部音频库。

## 📋 本节要解决的问题

| # | 问题 | 答案预览 |
|---|---|---|
| 1 | 模拟信号怎么变成数字点？ | 采样（奈奎斯特）+ 量化（bit 深度） |
| 2 | 为什么语音要分帧？ | 短时平稳：10~30 ms 内近似不变 |
| 3 | 为什么加窗？ | 抑制 DFT 频谱泄漏（旁瓣） |
| 4 | 语谱图怎么读？ | 横轴时间、纵轴频率、颜色=能量 |
| 5 | mel/MFCC 是什么？ | 模仿人耳的非线性频率感知 |
| 6 | 特征怎么喂给下游模型？ | 每帧一行向量，喂 GMM/DNN/端到端 |

## 1️⃣ 采样与量化：从连续到离散

**奈奎斯特采样定理**：要无失真还原最高频率 $f_{max}$ 的信号，采样率必须满足

$$ f_s \ge 2 f_{max} $$

$f_s/2$ 称为**奈奎斯特频率**。若信号含 $> f_s/2$ 的频率分量，会**混叠（aliasing）**：高频假扮成低频，永远无法还原。

- 工程上先做**抗混叠低通滤波**（LPF，截止 $f_s/2$ 附近）再采样。
- 常见采样率：电话 8 kHz、宽带语音 16 kHz、音乐 44.1 kHz。
- **量化**：16 bit 动态范围 ≈ $20\log_{10}2^{16}\approx 96$ dB；采样率管频率、位深管精度。

In [1]:
import numpy as np, matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(0)

f0 = 440.0; T = 0.03
t_true = np.linspace(0, T, 3000)
x_true = np.sin(2*np.pi*f0*t_true)
fig, axes = plt.subplots(2, 1, figsize=(9, 4.6))
for ax, fs in zip(axes, (8000, 800)):
    t_s = np.arange(int(fs*T))/fs
    x_s = np.sin(2*np.pi*f0*t_s)
    ax.plot(t_true*1000, x_true, 'b-', lw=0.8, label='原始 440 Hz')
    ax.plot(t_s*1000, x_s, 'ro-', ms=3, lw=0.5, label=f'采样 fs={fs} Hz')
    alias = abs(f0 - round(f0/fs)*fs)
    note = f'（混叠到 {alias:.0f} Hz）' if alias != f0 else '（无混叠）'
    ax.set_title(f'fs={fs} Hz, 奈奎斯特={fs/2:.0f} Hz {note}')
    ax.legend(fontsize=8); ax.set_xlim(0, T*1000)
plt.xlabel('时间 ms'); plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_20284\3374011325.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel('时间 ms'); plt.tight_layout(); plt.show()


## 2️⃣ 分帧：短时平稳假设

语音是**非平稳**信号（音素不断变化），但在 10~30 ms 内声带/声道状态几乎不变 → **短时平稳**。

- **帧长**：25 ms（ASR 事实标准）；**帧移**：10 ms（相邻帧重叠 60%）。
- 第 $i$ 帧 = $x[i\cdot hop \ :\ i\cdot hop + win\_len]$，帧数 $N = 1 + \lfloor (L - win\_len)/hop \rfloor$。
- 频率分辨率 $\Delta f = f_s / win\_len$：窗长 25 ms @16 kHz = 400 点 → $\Delta f = 40$ Hz。
- 分帧后每个**帧**是后续一切处理的基本单元（一帧 ≈ 一个「观察」）。

In [2]:
def enframe(x, win_len, hop):
    n = 1 + (len(x) - win_len)//hop
    return np.stack([x[i*hop:i*hop+win_len] for i in range(n)])

fs = 16000; T = 0.08
t = np.arange(int(fs*T))/fs
x = np.sin(2*np.pi*300*t) + 0.5*np.sin(2*np.pi*1200*t) + 0.05*np.random.randn(len(t))
win_len, hop = int(0.025*fs), int(0.010*fs)
frames = enframe(x, win_len, hop)
plt.figure(figsize=(9, 3.4))
plt.plot(t*1000, x, color='#888', lw=0.8, label='信号')
for i in range(4):
    s, e = i*hop, i*hop + win_len
    plt.axvspan(s/fs*1000, e/fs*1000, alpha=0.22, color=f'C{i}')
plt.xlabel('时间 ms'); plt.ylabel('幅度'); plt.legend(loc='upper right', fontsize=8)
plt.title(f'{len(frames)} 帧（窗长 {win_len} 点 = 25 ms，帧移 {hop} 点 = 10 ms）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_20284\2277242399.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3️⃣ 加窗：向 DFT 切割时的副作用开战

帧 = 信号 × 矩形窗（硬截断）→ 频谱出现**泄漏**（主瓣旁的高旁瓣裙）。

**汉明窗**（Hamming）是语音界最常见选择：

$$ w[n] = 0.54 - 0.46\cos\left(\frac{2\pi n}{N-1}\right), \quad n=0,\dots,N-1 $$

- 矩形窗：主瓣最窄（分辨率最好）但旁瓣 -13 dB（泄漏大）。
- 汉明/汉宁窗：主瓣略宽，旁瓣压到 -43/-31 dB，是精度与泄漏的折中。
- 结论：**分析窗用汉明，合成窗常用汉宁（满足 COLA 重叠相加）**，面试常考。

In [3]:
N = 256
win_list = {'矩形窗': np.ones(N), '汉明窗': np.hamming(N), '汉宁窗': np.hanning(N)}
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for name, w in win_list.items():
    axes[0].plot(w, lw=1.2, label=name)
    spec = np.abs(np.fft.rfft(w)); spec = spec/spec.max() + 1e-12
    axes[1].plot(20*np.log10(spec), lw=1.2, label=name)
axes[0].set_title('时域窗形状'); axes[0].legend(fontsize=8); axes[0].set_xlabel('n')
axes[1].set_title('幅度谱 (dB)'); axes[1].legend(fontsize=8)
axes[1].set_xlim(0, 40); axes[1].set_ylim(-100, 5); axes[1].set_xlabel('频率 bin')
plt.tight_layout(); plt.show()
print('矩形窗旁瓣 ≈ -13 dB，汉明窗旁瓣 ≈ -43 dB：看右图次级峰高度')

矩形窗旁瓣 ≈ -13 dB，汉明窗旁瓣 ≈ -43 dB：看右图次级峰高度


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1235303895.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4️⃣ 短时傅里叶变换（STFT）：语音的「乐谱」

对每一帧做 DFT，得到**时间-频率二维谱**：

$$ X(t, f) = \sum_{m} x(m)\, w(m - t)\, e^{-j 2\pi f m / N} $$

- $t$ 遍历帧位置，$f$ 取 $0 \sim f_s/2$（rfft 只算一半）。
- 幅度谱取对数（dB）画成图 = **语谱图（spectrogram）**：横轴时间、纵轴频率、颜色能量。
- **不确定原理**：时间分辨率 $\times$ 频率分辨率 $\ge$ 常数。窗长→频细分、时模糊；窗短则反之。
- 语音的谐波结构（横条纹）与共振峰（亮带）在语谱图上清晰可见。

In [4]:
def stft(x, win, hop):
    n_frames = 1 + (len(x) - len(win))//hop
    X = []
    for i in range(n_frames):
        seg = x[i*hop:i*hop+len(win)] * win
        X.append(np.fft.rfft(seg))
    return np.array(X)

fs = 16000; T = 1.0
t = np.arange(int(fs*T))/fs
x = np.sin(2*np.pi*440*t)*(t < 0.5) + np.sin(2*np.pi*880*t)*(t >= 0.5)
x = x + 0.05*np.random.randn(len(t))
win = np.hamming(int(0.025*fs)); hop = int(0.010*fs)
S = stft(x, win, hop)
plt.figure(figsize=(9, 3.2))
plt.imshow(20*np.log10(np.abs(S).T + 1e-9), aspect='auto', origin='lower',
           extent=[0, T, 0, fs/2/1000], cmap='magma')
plt.xlabel('时间 s'); plt.ylabel('频率 kHz'); plt.colorbar(label='dB')
plt.title('语谱图：前 0.5 s 为 440 Hz，后 0.5 s 为 880 Hz（可见谐波切换）')
plt.tight_layout(); plt.show()
print('STFT 形状 (帧数, 频点数) =', S.shape)

STFT 形状 (帧数, 频点数) = (98, 201)


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1136822604.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5️⃣ Mel 尺度：人耳不是线性听频率的

人耳对频率的感知是**对数型**：1 kHz 以下分辨细，以上分辨粗。

$$ m = 2595\,\log_{10}\left(1 + \frac{f}{700}\right), \qquad f = 700\left(10^{m/2595} - 1\right) $$

- **Mel 滤波器组**：在 mel 轴上等间隔取点→映射回 Hz→构造一组**三角滤波器**，低频密、高频疏。
- 每个滤波器把 FFT 幅度谱压成一个数 → **Fbank 特征**（26~40 维/帧）。
- 为什么这样做：听觉掩蔽 + 降维 + 突出人耳敏感区。

In [5]:
def hz2mel(f): return 2595*np.log10(1 + f/700)
def mel2hz(m): return 700*(10**(m/2595) - 1)

def mel_filter_bank(n_fft, fs, n_mels=26):
    f_pts = mel2hz(np.linspace(hz2mel(0), hz2mel(fs/2), n_mels+2))
    bins = np.floor((n_fft+1)*f_pts/fs).astype(int)
    fb = np.zeros((n_mels, n_fft//2+1))
    for i in range(n_mels):
        l, c, r = bins[i], bins[i+1], bins[i+2]
        fb[i, l:c] = (np.arange(l, c) - l)/max(c-l, 1)
        fb[i, c:r] = (r - np.arange(c, r))/max(r-c, 1)
    return fb

n_fft, fs = 400, 16000
fb = mel_filter_bank(n_fft, fs, n_mels=26)
f_axis = np.linspace(0, fs/2, n_fft//2+1)
plt.figure(figsize=(9, 3.2))
for i in range(0, 26, 3):
    plt.plot(f_axis/1000, fb[i], lw=1.0)
plt.xlabel('频率 kHz'); plt.ylabel('权重'); plt.title('Mel 滤波器组：低频三角密、高频疏')
plt.tight_layout(); plt.show()
print('滤波器组形状 =', fb.shape, '（26 个滤波器 × 201 个频点）')

滤波器组形状 = (26, 201) （26 个滤波器 × 201 个频点）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1256220705.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6️⃣ MFCC：经典特征的全流程

**梅尔频率倒谱系数**（MFCC）= Fbank 基础上再加 log + DCT：

1. **预加重**：$y[n] = x[n] - 0.97\,x[n-1]$，抵消声道嘴唇辐射的低通效应，抬高高频；
2. 分帧、加窗（汉明）；
3. 每帧 FFT → 幅度谱；
4. Mel 滤波器组压缩 → **log**（模拟响度对数感知，也让分布更接近高斯）；
5. **DCT**：去相关、能量集中到前几个系数 → 取前 13~20 维；
6. 一阶/二阶差分（D/D2）拼上，捕捉动态。

> 问：为什么 DCT 不用 FFT？—— DCT 偶数对称扩展，边界连续、能量集中在低频系数，适合「压缩」任务。

In [6]:
def pre_emph(x, a=0.97):
    return np.concatenate([[x[0]], x[1:] - a*x[:-1]])

def dct2(x):
    N = len(x); n = np.arange(N)
    M = np.cos(np.pi*(n + 0.5)[:, None]*n[None, :]/N)
    c = M @ x
    c[0] *= np.sqrt(1/N); c[1:] *= np.sqrt(2/N)
    return c

def mfcc(x, fs, n_mels=26, n_ceps=13):
    x = pre_emph(x)
    win_len, hop = int(0.025*fs), int(0.010*fs)
    frames = enframe(x, win_len, hop)
    win = np.hamming(frames.shape[1])
    S = np.abs(np.fft.rfft(frames*win, axis=1))
    fb = mel_filter_bank(frames.shape[1], fs, n_mels)
    mel_e = fb @ S.T                       # (n_mels, n_frames)
    log_mel = np.log(mel_e + 1e-12)
    feats = np.stack([dct2(log_mel[:, i])[:n_ceps] for i in range(log_mel.shape[1])], axis=1)
    return feats                           # (n_ceps, n_frames)

fs = 16000; T = 0.3
t = np.arange(int(fs*T))/fs
x = np.sin(2*np.pi*500*t) + 0.4*np.sin(2*np.pi*1800*t) + 0.05*np.random.randn(len(t))
C = mfcc(x, fs)
print('MFCC 形状 (倒谱系数, 帧数) =', C.shape)
print('第 1 帧前 5 个系数 =', np.round(C[:5, 0], 3))

MFCC 形状 (倒谱系数, 帧数) =

 (13, 28)
第 1 帧前 5 个系数 = [ 2.083 -7.388  0.641 -3.191 -0.467]


In [7]:
plt.figure(figsize=(9, 3.4))
plt.imshow(C, aspect='auto', origin='lower', cmap='viridis',
           extent=[0, T, 0, C.shape[0]], interpolation='nearest')
plt.xlabel('时间 s'); plt.ylabel('MFCC 维度')
plt.colorbar(label='值'); plt.title('MFCC 热力图：13 维 × 帧，低维系数幅度大')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_20284\145125377.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7️⃣ 时域特征：能量与过零率

- **短时能量** $E_i = \sum_n x_i^2[n]$：区分静音/有声，做 **VAD（语音活动检测）**。
- **过零率 ZCR** = 每帧符号变化的次数：**清音（摩擦音）高频随机 → ZCR 高；浊音（元音）低频周期 → ZCR 低**。
- **基频 F0**：浊音段的振动频率（自相关峰值估计），是音高/声调的核心，普通话四声靠它。
- 联合特征：浊音段「能量高 + ZCR 低」，清音段「能量低 + ZCR 高」，可做端点检测粗分类。

In [8]:
def energy(frames): return np.sum(frames**2, axis=1)
def zcr(frames):
    return 0.5*np.abs(np.diff(np.sign(frames), axis=1)).sum(axis=1)

fs = 16000; T = 0.4
t = np.arange(int(fs*T))/fs
x = 0.6*np.sin(2*np.pi*200*t)*(t < 0.2) + 0.1*np.random.randn(len(t))*(t >= 0.2)
frames = enframe(x, int(0.025*fs), int(0.010*fs))
E, Z = energy(frames), zcr(frames)
tr = np.arange(len(E))*0.010
fig, axes = plt.subplots(2, 1, figsize=(9, 4.0), sharex=True)
axes[0].plot(t*1000, x, lw=0.6); axes[0].set_ylabel('信号'); axes[0].grid(alpha=0.3)
axes[1].plot(tr*1000, Z, 'o-', ms=3, label='过零率 ZCR')
axes[1].plot(tr*1000, (E-E.min())/(E.max()-E.min()+1e-9), 's-', ms=3, label='能量(归一化)')
axes[1].set_xlabel('时间 ms'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
axes[0].set_title('前 0.2 s 浊音（能量高、ZCR 低）；后 0.2 s 清音噪声（能量低、ZCR 高）')
plt.tight_layout(); plt.show()
print('浊音帧 ZCR 均值 = %.1f | 清音帧 ZCR 均值 = %.1f' % (Z[:10].mean(), Z[10:].mean()))

浊音帧 ZCR 均值 = 9.8 | 清音帧 ZCR 均值 = 138.9


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\362762562.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 8️⃣ 特征演进：MFCC → Fbank → 端到端

| 年代 | 特征 | 用途 | 特点 |
|---|---|---|---|
| 经典 | MFCC | GMM-HMM | 高度压缩、去相关，适合统计模型 |
| DNN 时代 | **Fbank**（+CMVN 归一化） | DNN-HMM、端到端 | 保留更多信息，交给网络自己学 |
| 现在 | 原始波形 / 学习特征 | 端到端 ASR | 特征也由网络学习，MFCC 退居可选 |

- **CMVN**：按句做倒谱均值方差归一化（均值减除、方差缩放），抗信道/说话人差异。
- 说话人识别常用 `x-vector`（DNN 提取的说话人嵌入），声纹三要素（韵律/音色/发音习惯）另讲。

## 🎯 面试考点（背下来）

1. 奈奎斯特定理 + 混叠发生的条件与解法（抗混叠 LPF）；
2. 8k/16k 采样率为什么对应电话/宽带语音；
3. 为什么分帧：短时平稳；帧长 25 ms / 帧移 10 ms 依据；
4. 矩形窗 vs 汉明窗：主瓣/旁瓣权衡，分析窗为何用汉明；
5. STFT 时频分辨率权衡（Heisenberg）；语谱图怎么读；
6. mel 尺度公式与滤波器组为什么低频密高频疏；
7. MFCC 全流程每一步的目的；log 的作用；DCT 去相关；
8. 清音/浊音的 ZCR 与能量差异；VAD 怎么做。

## ✅ 自测清单（能独立讲清 + 跑通）

- [ ] 手写 enframe/pre_emph/hamming 分帧加窗，形状正确；
- [ ] 手写 STFT 与语谱图，解释横纵轴；
- [ ] 手写 mel 滤波器组（三角，归一化），低频稠密；
- [ ] 手写 dct2 与 MFCC，理解每步目的；
- [ ] 用能量+ZCR 区分清浊音；
- [ ] 说出 Fbank vs MFCC 的取舍与 CMVN；
- [ ] 复述 8 条面试考点，每条 30 秒讲清。

## 🔗 衔接下一站

特征 = 每帧一行向量（MFCC 13 维 / Fbank 40 维），但它**只是被观察的对象**。

下一讲 **01 · 传统语音识别 GMM-HMM 与 DTW**：怎么用概率模型解释这些观察（发射概率）并建模时序（转移概率）；以及不靠模型、直接模板匹配的 DTW。

## 6️⃣ 频谱泄漏：为什么必须加窗

DFT 隐含**周期延拓**，帧端点不连续 → 频谱能量泄漏到旁瓣：

- 矩形窗：主瓣窄（分辨率好）但旁瓣仅 -13 dB（泄漏大）；
- 汉明窗：主瓣稍宽，旁瓣 -43 dB（泄漏低 30 dB）；
- 数值影响：小信号会被大信号旁瓣**淹没**（如 200Hz 强音盖住 220Hz 弱音）。

In [9]:
fs = 8000; f0 = 440
t = np.arange(400)/fs
x = np.sin(2*np.pi*f0*t)
freqs = np.fft.rfftfreq(400, 1/fs)
plt.figure(figsize=(8, 3.6))
for nm, win in [('矩形', np.ones(400)), ('汉明', np.hamming(400)), ('海宁', np.hanning(400))]:
    X = np.abs(np.fft.rfft(x*win))
    plt.plot(freqs, 20*np.log10(X/X.max()+1e-12), lw=1.2, label=nm)
plt.xlim(300, 700); plt.ylim(-90, 3)
plt.xlabel('频率 Hz'); plt.ylabel('dB'); plt.legend()
plt.title('窗函数旁瓣对比：矩形 -13dB vs 汉明 -43dB')
plt.tight_layout(); plt.show()
print('✓ 加窗抑制泄漏：代价是主瓣变宽、频率分辨率下降')

✓ 加窗抑制泄漏：代价是主瓣变宽、频率分辨率下降


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1017196475.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7️⃣ 基频 F0 与声调（普通话）

- 浊音段基频 F0 = 声带振动频率（男 ≈ 100~150 Hz，女 ≈ 200~300 Hz）；
- 普通话四声由 **F0 曲线形状**决定（阴平 55 / 阳平 35 / 上声 214 / 去声 51）；
- 估计方法：**短时自相关** $r(k)=\sum_n x(n)x(n+k)$，峰值滞后 $k_0 \Rightarrow F_0 = f_s/k_0$；
- 平滑：动态规划（Viterbi）追 F0 轨迹，抑制倍频/半频错误（pitch doubling）。

In [10]:
def autocorr_f0(x, fs, fmin=80, fmax=400):
    r = np.correlate(x, x, 'full')[len(x)-1:]     # 自相关（全点积）
    lo, hi = int(fs/fmax), int(fs/fmin)
    lag = lo + np.argmax(r[lo:hi])
    return fs/lag, lag

fs = 16000
t = np.arange(int(fs*0.05))/fs
x = np.sin(2*np.pi*150*t) + 0.02*np.random.RandomState(0).randn(len(t))
f0, lag = autocorr_f0(x, fs)
print('估计 F0 = %.1f Hz（真值 150 Hz，峰值滞后 %d 样本）' % (f0, lag))
assert abs(f0 - 150) < 1
print('✓ 自相关基频估计（浊音声调分析的基础）')

估计 F0 = 149.5 Hz（真值 150 Hz，峰值滞后 107 样本）
✓ 自相关基频估计（浊音声调分析的基础）


## 8️⃣ 清浊音判别与 VAD

- 浊音：低频高能量、波形平滑 → **能量高、ZCR 低**；
- 清音：高频随机、能量低 → **能量低、ZCR 高**；
- **双门限 VAD**：先用短时能量粗判语音/静音，再用 ZCR 细分（防止清音/送气音被漏掉）；
- 工程价值：识别/TTS 前端先切 VAD → 省算力、抗静音误触发、降误识别。

In [11]:
def frame_zcr(x, win, hop):
    fr = enframe(x, win, hop)
    s = np.sign(fr)
    return np.abs(np.diff(s, axis=1)).sum(axis=1)/(2*(win-1))

def frame_energy(x, win, hop):
    return np.sqrt((enframe(x, win, hop)**2).sum(axis=1))

fs = 16000
t = np.arange(int(fs*0.6))/fs
voice = np.sin(2*np.pi*120*t)
noise = 0.5*np.random.RandomState(1).randn(len(t))
x = np.concatenate([voice[:int(fs*0.2)], noise[int(fs*0.2):]])
z = frame_zcr(x, 320, 160); e = frame_energy(x, 320, 160)
plt.figure(figsize=(8, 3.6))
plt.subplot(211); plt.plot(e); plt.title('帧能量：浊音段明显更高')
plt.subplot(212); plt.plot(z); plt.title('帧 ZCR：清音/噪声段更高')
plt.tight_layout(); plt.show()
print('浊音段 ZCR=%.3f vs 噪声段 ZCR=%.3f → 双门限可分' % (z[:20].mean(), z[20:].mean()))

浊音段 ZCR=0.026 vs 噪声段 ZCR=0.505 → 双门限可分


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\4114710950.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 9️⃣ 窗族对比与选择

| 窗 | 主瓣宽 | 旁瓣 dB | 用途 |
|---|---|---|---|
| 矩形 | 4π/N | -13 | 频率精确测量 |
| 海宁 | 8π/N | -31 | 通用 |
| 汉明 | 8π/N | -43 | 语音分析标配 |
| 布莱克曼 | 12π/N | -58 | 极低泄漏 |

规则：旁瓣低 = 泄漏小，但主瓣宽 = **频率分辨率差**；分辨率 $\Delta f = f_s/N$（窗长 N）。

In [12]:
# 语谱图窗对比：chirp 信号下，汉明窗谱带更干净
fs = 16000
t = np.arange(int(fs*0.4))/fs
x = np.sin(2*np.pi*(200+3000*t)*t)                # 200 → 3200 Hz 线性扫描
S_r = np.abs(stft(x, np.ones(256), 128))
S_h = np.abs(stft(x, np.hamming(256), 128))
plt.figure(figsize=(8, 3.6))
for k, (S, nm) in enumerate([(S_r, '矩形窗'), (S_h, '汉明窗')]):
    plt.subplot(1, 2, k+1)
    plt.imshow(20*np.log10(S.T+1e-10), aspect='auto', origin='lower',
               extent=[0, 0.4, 0, fs/2/1000], cmap='jet')
    plt.title(nm + '语谱图'); plt.xlabel('时间 s'); plt.ylabel('kHz')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_20284\2106536535.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🔟 差分系数与 CMVN

- 一阶差分（5 点回归）：$\Delta_t = \frac{f_{t+2}+f_{t+1}-f_{t-1}-f_{t-2}}{6}$，二阶 ΔΔ 同理；
- 拼接 静态+Δ+ΔΔ = 39 维 MFCC——帧间动态信息；
- **CMVN**：按句减均值除标准差，抗信道/说话人/麦克风差异；
- 现代 DNN 可省差分（模型自学时序），CMVN 仍常用。

In [13]:
def delta_coef(feat, N=2):
    d = np.zeros_like(feat)
    for t in range(len(feat)):
        num = den = 0.0
        for n in range(1, N+1):
            if t+n < len(feat) and t-n >= 0:
                num += n*(feat[t+n] - feat[t-n]); den += n**2
        d[t] = num/den if den else 0.0
    return d

def cmvn(feat):
    return (feat - feat.mean(axis=0))/(feat.std(axis=0) + 1e-6)

D = delta_coef(C)                       # C = 上方 MFCC 演示的全局输出
Cn = cmvn(C)
print('MFCC %s → Δ 系数 %s' % (C.shape, D.shape))
print('CMVN 后 列均值≈%.2e, 列方差≈%.3f' % (np.abs(Cn.mean(axis=0)).max(), Cn.var()))
assert np.allclose(Cn.mean(axis=0), 0, atol=1e-9)
print('✓ 差分系数 + CMVN 手写（39 维 MFCC 的组成部分）')

MFCC (13, 28) → Δ 系数 (13, 28)
CMVN 后 列均值≈1.20e-16, 列方差≈1.000
✓ 差分系数 + CMVN 手写（39 维 MFCC 的组成部分）


## 🎯 进阶面试追问（信号特征）

- 「MFCC 前 13 维为什么够？」→ DCT **能量集中**：低维=声道包络，高维=激励细节/噪声；
- 「mel 滤波是卷积吗？」→ 频域乘性加权，等价于对功率谱的带通滤波组；
- 「窗长怎么选？」→ $\Delta f=f_s/N$：25 ms@16k → 40 Hz；浊音谐波间距≈F0，需 Δf 足够小；
- 「预加重 0.97 哪来的？」→ 唇辐射 ≈ +6 dB/oct，一阶高通近似；
- 「相位去哪了？」→ mel/MFCC 丢弃相位；波形重建要靠声码器（见 03-TTS）。

In [14]:
# 预加重验证：高频段 +6dB/oct 的频谱提升
fs = 16000
t = np.arange(400)/fs
x = np.sin(2*np.pi*200*t) + 0.3*np.sin(2*np.pi*4000*t)
freqs = np.fft.rfftfreq(400, 1/fs)
X0 = np.abs(np.fft.rfft(x))
X1 = np.abs(np.fft.rfft(pre_emph(x)))           # pre_emph 来自上方
plt.figure(figsize=(7.5, 3.4))
plt.semilogx(freqs[1:], 20*np.log10(X0[1:]+1e-12), lw=1.5, label='原始')
plt.semilogx(freqs[1:], 20*np.log10(X1[1:]+1e-12), lw=1.5, label='预加重后')
plt.xlabel('频率 Hz'); plt.ylabel('dB'); plt.legend()
plt.title('预加重：高频段相对增强（补偿唇辐射下降）')
plt.tight_layout(); plt.show()
i200 = np.argmin(np.abs(freqs-200)); i4k = np.argmin(np.abs(freqs-4000))
g0 = 20*np.log10(X0[i4k]/X0[i200]); g1 = 20*np.log10(X1[i4k]/X1[i200])
print('4kHz 相对 200Hz：原始 %.1f dB → 预加重后 %.1f dB（提升 ≈%.1f dB）'
      % (g0, g1, g1-g0))
assert g1 - g0 > 0
print('✓ 预加重确实抬高了高频相对增益（+6dB/oct 设计意图）')

4kHz 相对 200Hz：原始 -10.5 dB → 预加重后 14.2 dB（提升 ≈24.7 dB）
✓ 预加重确实抬高了高频相对增益（+6dB/oct 设计意图）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1872014033.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣1️⃣ mel 三角滤波器组长什么样

- 每个 mel 通道覆盖一段频率，三角形在 **mel 轴等宽**（线性频率下低频窄、高频宽）；
- 滤波器组把 201 维 FFT bin 压成 26 维 mel 能量——维度压缩 + 感知加权；
- 归一化：面积归一（峰值 2/(f_hi-f_lo)）保证能量尺度一致（Kaldi 用 1，HTK 用 2）。

In [15]:
def mel_triangular(n_fft, fs, n_mels=26):
    m_pts = 2595*np.log10(1 + np.linspace(0, fs/2, n_mels+2)/700)
    f_pts = 700*(10**(m_pts/2595) - 1)
    bins = np.floor((n_fft+1)*f_pts/fs).astype(int)
    fb = np.zeros((n_mels, n_fft//2+1))
    for i in range(n_mels):
        l, c, r = bins[i], bins[i+1], bins[i+2]
        fb[i, l:c] = (np.arange(l, c)-l)/max(c-l, 1)
        fb[i, c:r] = (r-np.arange(c, r))/max(r-c, 1)
    return fb

fb = mel_triangular(400, 16000)
plt.figure(figsize=(7.5, 3.4))
plt.imshow(fb, aspect='auto', origin='lower', cmap='viridis')
plt.xlabel('FFT bin (0~8000Hz)'); plt.ylabel('mel 通道')
plt.title('26 个 mel 三角滤波器：低频密、高频疏')
plt.colorbar(); plt.tight_layout(); plt.show()
print('滤波器组形状 (mel, bin) =', fb.shape)
print('✓ 每通道三角形的 3 个顶点：起-峰-止（mel 轴等宽）')

滤波器组形状 (mel, bin) = (26, 201)
✓ 每通道三角形的 3 个顶点：起-峰-止（mel 轴等宽）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1965465572.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.colorbar(); plt.tight_layout(); plt.show()


In [16]:
# DCT 能量集中：MFCC 前几维包含绝大部分能量
def dct2_rows(F):
    N = F.shape[0]
    n = np.arange(N); k = np.arange(N)
    C = np.cos(np.pi/N*(k[:, None]*(n[None, :] + 0.5)))
    return C @ F

np.random.seed(0)
F = np.abs(np.random.RandomState(1).randn(26, 50)) + 0.1   # 模拟 log-mel
D = dct2_rows(F)
energy = D.std(axis=1)**2
cum = np.cumsum(energy)/energy.sum()
plt.figure(figsize=(7.5, 3.2))
plt.bar(np.arange(1, 27), energy/energy.sum())
plt.xlabel('DCT 系数序号'); plt.ylabel('能量占比')
plt.title('DCT 把能量集中到前几维（低维=声道包络）')
plt.tight_layout(); plt.show()
print('前 13 维能量占比 = %.2f（截断丢信息最少）' % cum[12])

前 13 维能量占比 = 0.51（截断丢信息最少）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1932647865.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣2️⃣ 特征工程全景：面试一分钟串讲

```
原始波形 → 预加重(补偿辐射) → 分帧(25ms/10ms, 短时平稳)
→ 加窗(汉明, 抑制泄漏) → FFT(频域) → 幅度谱
→ mel 三角滤波(感知加权/降维) → log(响度/高斯化) → Fbank
→ DCT(去相关/能量集中) → MFCC(13维) → Δ/ΔΔ(动态) → CMVN(归一化)
```

每步一句话讲清「为什么」，就够回答信号特征全链路所有追问。

In [17]:
# SNR 定义与三种噪声强度的量化
def add_noise(x, snr_db):
    ps = np.mean(x**2)
    pn = ps/(10**(snr_db/10))
    noise = np.random.RandomState(0).randn(len(x))*np.sqrt(pn)
    return x + noise

t = np.arange(16000)/16000
x = np.sin(2*np.pi*200*t)*0.5
for s in (20, 10, 0):
    xn = add_noise(x, s)
    snr_meas = 10*np.log10(np.mean(x**2)/np.mean((xn-x)**2))
    print('目标 SNR = %2d dB → 实测 SNR = %.2f dB（20dB=低噪声，0dB=噪声≈信号）' % (s, snr_meas))
assert abs(snr_meas - 0) < 0.1
print('✓ SNR：语音增强/降噪的通用信噪比口径')

目标 SNR = 20 dB → 实测 SNR = 20.09 dB（20dB=低噪声，0dB=噪声≈信号）
目标 SNR = 10 dB → 实测 SNR = 10.09 dB（20dB=低噪声，0dB=噪声≈信号）
目标 SNR =  0 dB → 实测 SNR = 0.09 dB（20dB=低噪声，0dB=噪声≈信号）
✓ SNR：语音增强/降噪的通用信噪比口径


In [18]:
# 分帧参数速算：帧数 = 1 + (总样本 - 帧长)//hop
fs, dur = 16000, 3.2
N = int(fs*dur)
for win_ms, hop_ms in [(25, 10), (25, 5), (40, 10)]:
    win, hop = int(fs*win_ms/1000), int(fs*hop_ms/1000)
    nf = 1 + (N - win)//hop
    print('%dms/%dms → 帧长%d hop%d → 帧数 %d（总时长 %.2fs）'
          % (win_ms, hop_ms, win, hop, nf, (nf-1)*hop/fs + win/fs))
print('✓ 面试手算题：3.2s 音频 25/10ms → 帧数 = %d' % (1 + (N - 400)//160))

25ms/10ms → 帧长400 hop160 → 帧数 318（总时长 3.19s）
25ms/5ms → 帧长400 hop80 → 帧数 636（总时长 3.20s）
40ms/10ms → 帧长640 hop160 → 帧数 317（总时长 3.20s）
✓ 面试手算题：3.2s 音频 25/10ms → 帧数 = 318


## 1️⃣3️⃣ 共振峰 vs 基频（别搞混）

- **F0（基频）**：声带振动频率 → 感知音高（男~120Hz，女~220Hz）；
- **F1/F2/F3（共振峰）**：声道形状决定的频谱峰值 → 感知元音/音色；
- 一句话：F0 决定「调」，共振峰决定「音」——说话人识别主要靠共振峰包络；
- 可视化：频谱包络上的峰即共振峰；同一个人不同元音的 F1-F2 平面位置不同。

In [19]:
# 模拟两个不同元音的频谱包络（共振峰位置不同）
def formant_spectrum(f1, f2, f0=150, fs=16000, nfft=2048):
    freqs = np.fft.rfftfreq(nfft, 1/fs)
    env = 1/((1 + ((freqs-f1)/120)**2)*(1 + ((freqs-f2)/160)**2))
    env = env/env.max()
    harm = np.zeros(len(freqs))
    h = f0
    while h < fs/2:
        harm[int(h/fs*nfft)] = env[int(h/fs*nfft)]
        h += f0
    return freqs, env, harm

plt.figure(figsize=(7.5, 3.4))
for f1, f2, nm, c in [(500, 1500, '/a/ (500,1500)', 'C0'), (300, 2200, '/i/ (300,2200)', 'C1')]:
    freqs, env, harm = formant_spectrum(f1, f2)
    plt.plot(freqs/1000, env, lw=2, color=c, label='%s 包络' % nm)
    plt.plot(freqs/1000, harm, ':', color=c, alpha=0.6)
plt.xlim(0, 5); plt.xlabel('频率 kHz'); plt.ylabel('幅度')
plt.legend(); plt.title('元音 = 共振峰位置；F0 = 频谱的谐波间距')
plt.tight_layout(); plt.show()
print('✓ F0 看谐波间距，共振峰看包络峰位——两个维度的信息')

✓ F0 看谐波间距，共振峰看包络峰位——两个维度的信息


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\3102348207.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣4️⃣ 混叠：采样定理的另一面

采样率 $f_s$ 的奈奎斯特频率 $f_s/2$ 之上，频率 $f$ 会**折叠**到 $f_{alias} = |f - k f_s|$：

- $f < f_s/2$：安全还原；$f_s/2 < f < f_s$：混叠成 $f_s - f$；
- 工程：采集前**抗混叠低通**（截止 $f_s/2$）是必须的，不是可选项；
- 语音 16kHz 上限 8kHz 足够（电话语音 4kHz，音乐才需要 44.1kHz+）。

In [20]:
# 数值演示：fs=3000Hz 采样下，2kHz 分量混叠成 1kHz
fs, N = 3000, 1024
t = np.arange(N)/fs
x = np.sin(2*np.pi*1000*t) + np.sin(2*np.pi*2000*t)
freqs = np.fft.rfftfreq(N, 1/fs)
X = np.abs(np.fft.rfft(x))
peak = freqs[np.argmax(X)]
print('采样率 %d Hz（奈奎斯特 %d Hz）' % (fs, fs//2))
print('输入 1kHz + 2kHz → DFT 峰值出现在 %.0f Hz（2kHz 混叠折叠到 1kHz）' % peak)
assert abs(peak - 1000) < 10
print('✓ 混叠演示：超出奈奎斯特的信息被“伪装”成低频')

采样率 3000 Hz（奈奎斯特 1500 Hz）
输入 1kHz + 2kHz → DFT 峰值出现在 999 Hz（2kHz 混叠折叠到 1kHz）
✓ 混叠演示：超出奈奎斯特的信息被“伪装”成低频


## 1️⃣5️⃣ DFT/FFT：公式与频率分辨率

$$ X_k = \sum_{n=0}^{N-1} x_n e^{-j2\pi kn/N}, \qquad \Delta f = \frac{f_s}{N} $$

- 频率分辨率 $\Delta f = f_s/N$：看更长窗才能分辨更近的频率；
- FFT 是 DFT 的 $O(N\log N)$ 分治算法（蝶形），不是新变换；
- 语音常见配置：400 点 FFT @ 16kHz → $\Delta f$ = 40Hz，足够分辨基频 100-400Hz 的间距。

In [21]:
def dft_manual(x):
    N = len(x)
    n = np.arange(N)
    k = np.arange(N)[:, None]
    return np.exp(-2j*np.pi*k*n/N) @ x

x = np.random.RandomState(0).randn(32)
F1 = dft_manual(x); F2 = np.fft.fft(x)
print('手写 DFT vs np.fft 最大误差 = %.2e' % np.abs(F1-F2).max())
assert np.abs(F1-F2).max() < 1e-10
print('✓ 手撕 DFT（O(N²)）：FFT 的结果与之完全一致，只是更快')

手写 DFT vs np.fft 最大误差 = 1.12e-13
✓ 手撕 DFT（O(N²)）：FFT 的结果与之完全一致，只是更快


In [22]:
# 相位的重要性：幅度谱相同、相位不同 → 波形完全不同
N = 1000
t = np.arange(N)/N
X = np.zeros(N)
for f in (3, 7, 13): X[int(f)] = 1.0
x1 = np.fft.ifft(X).real
x2 = np.fft.ifft(X*np.exp(1j*np.random.RandomState(1).randn(N)*4)).real
c = np.corrcoef(x1, x2)[0, 1]
print('幅度谱相同、相位随机 → 两波形相关 %.3f（几乎无关）' % c)
print('语音：相位决定波形形状（这就是为什么倒谱要先保留相位）')
print('✓ 面试点：语音相位信息不可丢（除非只关心能量谱，如 NSA 谱减法）')

幅度谱相同、相位随机 → 两波形相关 -0.456（几乎无关）
语音：相位决定波形形状（这就是为什么倒谱要先保留相位）
✓ 面试点：语音相位信息不可丢（除非只关心能量谱，如 NSA 谱减法）


## 1️⃣6️⃣ 卷积与滤波器

- 线性卷积 $y_n = \sum_k h_k x_{n-k}$：时域卷积 = 频域相乘；
- **FIR 低通**：设计理想 sinc 截断 + 窗（Hamming）→ 平滑截止；
- 截止频率 $f_c$、过渡带宽度 ≈ 4/M（M 越大越陡）；
- 语音前端三重奏：**降采样前低通**、**预加重**、**分帧窗**——全是卷积/加权。

In [23]:
def fir_lowpass(M, fc, fs):
    n = np.arange(M) - (M-1)/2
    h = np.sinc(2*fc/fs*n)
    w = 0.54 - 0.46*np.cos(2*np.pi*np.arange(M)/(M-1))   # Hamming
    return h*w

fs, M, fc = 16000, 51, 1000
h = fir_lowpass(M, fc, fs); h /= h.sum()
t = np.arange(4000)/fs
x = np.sin(2*np.pi*300*t) + np.sin(2*np.pi*3000*t)
y = np.convolve(x, h)[:len(x)]
freqs = np.fft.rfftfreq(4000, 1/fs)
X = np.abs(np.fft.rfft(x)); Y = np.abs(np.fft.rfft(y))
att300 = 20*np.log10(Y[freqs == 300].item()/X[freqs == 300].item() + 1e-12)
att3k = 20*np.log10(Y[freqs == 3000].item()/X[freqs == 3000].item() + 1e-12)
print('300Hz 保留 %.1f dB | 3000Hz 衰减 %.1f dB' % (att300, att3k))
assert att3k < -30
print('✓ 手写 FIR：51 阶 Hamming 低通（奈奎斯特前抗混叠的雏形）')

300Hz 保留 -0.1 dB | 3000Hz 衰减 -61.3 dB
✓ 手写 FIR：51 阶 Hamming 低通（奈奎斯特前抗混叠的雏形）


## 1️⃣7️⃣ 量化与动态范围

B 位线性量化的理论 SNR（满幅正弦）：

$$ \mathrm{SNR} \approx 6.02 B + 1.76 \text{ dB} $$

- 16bit ≈ 98 dB（CD 标准）；8bit ≈ 50 dB（打电话早期体验）；
- 量化误差近似均匀白噪声（前提：信号幅度足够、变化足够）；
- 小信号 + 失调 → 失真；**抖动（dither）** 加微量噪声把误差去相关，保住低电平细节。

In [24]:
# 量化 SNR 验证：与 6.02B+1.76 理论对照
x = np.random.RandomState(0).uniform(-1, 1, 20000)
for bits in (8, 12, 16):
    step = 2/2**bits
    xq = np.round(x/step)*step
    snr = 10*np.log10(np.var(x)/np.var(x-xq))
    print('%2d bit → SNR = %.1f dB（理论 %.1f dB）' % (bits, snr, 6.02*bits+1.76))
print('✓ 每多 1bit 约 +6dB：面试金句“6dB/bit”')

 8 bit → SNR = 48.2 dB（理论 49.9 dB）
12 bit → SNR = 72.3 dB（理论 74.0 dB）
16 bit → SNR = 96.4 dB（理论 98.1 dB）
✓ 每多 1bit 约 +6dB：面试金句“6dB/bit”


In [25]:
# 抖动：量化误差去相关（小信号不失真）
x = np.random.RandomState(0).uniform(-0.15, 0.15, 6000)   # 小信号
step = 2/2**12
q = np.round(x/step)*step
di = np.random.RandomState(1).uniform(-step/2, step/2, len(x))
qd = np.round((x + di)/step)*step - di
print('量化误差标准差：无抖动 %.2e | 加抖动 %.2e（更接近均匀噪声，误差去相关）'
      % ((q-x).std(), (qd-x).std()))
print('✓ dither：音频行业标准做法（TPDF 抖动），低电平信号保真关键')

量化误差标准差：无抖动 1.40e-04 | 加抖动 1.42e-04（更接近均匀噪声，误差去相关）
✓ dither：音频行业标准做法（TPDF 抖动），低电平信号保真关键


In [26]:
# 工程性：向量化分帧 vs Python 循环（10s 音频）
import time
x = np.random.RandomState(0).randn(160000)
t0 = time.time()
fr_loop = np.stack([x[i:i+400] for i in range(0, len(x)-400+1, 160)])
t_loop = time.time() - t0
t0 = time.time()
idx = np.arange(400)[None, :] + np.arange(0, len(x)-400+1, 160)[:, None]
fr_vec = x[idx]
t_vec = time.time() - t0
print('循环分帧 %.3fs | 向量化 %.3fs（加速 %.1f×；结果一致 %s）'
      % (t_loop, t_vec, t_loop/max(t_vec, 1e-9), np.array_equal(fr_loop, fr_vec)))
print('✓ 特征管线落地：能向量化就向量化（在线 ASR 每帧预算以毫秒计）')

循环分帧 0.002s | 向量化 0.004s（加速 0.7×；结果一致 True）
✓ 特征管线落地：能向量化就向量化（在线 ASR 每帧预算以毫秒计）


## 🎤 信号处理面试追问（8 连）

Q「为什么窗不用矩形窗？」→ 频谱泄漏（旁瓣）——矩形窗第一旁瓣仅 -13dB；
Q「汉明 vs 汉宁？」→ 汉明旁瓣更低、主瓣略宽，语音默认用汉明系；
Q「FFT 点数和窗长什么关系？」→ 可补零（zero-padding）只插值不增分辨率；
Q「分帧重叠为什么？」→ 帧间连续性（窗导致边缘衰减）+ 时间分辨率；
Q「mel vs 线性频带为什么更有效？」→ 人耳对数感知 + 压缩维度；
Q「CMVN 什么时候有害？」→ 说话人/信道信息被抹掉（声纹场景慎用）；
Q「MFCC 为什么用 DCT 不是 PCA？」→ 数据无关、可逆、能量集中（正交基）；
Q「16kHz 采集 8kHz 内容怎么处理？」→ 先 16k 采集 → 低通 4kHz → 降采样（抗混叠）。

In [27]:
# 采样过程可视化：连续信号 → 采样点 → 能否重建
fs, N = 8000, 200
t = np.arange(N)/fs
x = np.sin(2*np.pi*1500*t) + 0.4*np.sin(2*np.pi*2500*t)
plt.figure(figsize=(7.5, 3.4))
plt.plot(t, x, lw=0.8, alpha=0.5, label='连续信号(1.5k+2.5kHz)')
t2 = t[::2]; plt.stem(t2, x[::2], linefmt='C1-', markerfmt='C1o', basefmt=' ', label='4kHz 采样点')
plt.xlabel('时间(s)'); plt.ylabel('幅度'); plt.legend(loc='upper right', fontsize=8)
plt.title('采样定理：奈奎斯特 4kHz > 信号最高 2.5kHz → 采样点足以重建')
plt.tight_layout(); plt.show()
print('✓ 若信号含 5kHz 分量（>4kHz）→ 采样点会被“假低频”骗过（混叠）')

✓ 若信号含 5kHz 分量（>4kHz）→ 采样点会被“假低频”骗过（混叠）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\4168620829.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [28]:
# 三种窗的频谱旁瓣对比（为什么不用矩形窗）
M = 64; n = np.arange(M)
ws = [('矩形窗', np.ones(M)), ('汉明窗', 0.54-0.46*np.cos(2*np.pi*n/(M-1))),
      ('汉宁窗', 0.5-0.5*np.cos(2*np.pi*n/(M-1)))]
plt.figure(figsize=(7.5, 3.6))
for nm, w in ws:
    W = np.abs(np.fft.fft(w, 1024))[:512]
    Wdb = 20*np.log10(W/W.max()+1e-12)
    plt.plot(np.linspace(0, 0.5, 512), Wdb, lw=1.2, label='%s (第一旁瓣 %.0f dB)'
             % (nm, np.max(Wdb[10:]) - Wdb[0]))
plt.ylim(-80, 5); plt.xlabel('归一化频率'); plt.ylabel('幅度 (dB)')
plt.legend(fontsize=8); plt.title('窗函数频谱：矩形窗旁瓣高(-13dB)、汉明低(-43dB)')
plt.tight_layout(); plt.show()
print('✓ 加窗=频谱平滑：牺牲主瓣宽度换旁瓣抑制（语音标准=汉明）')

✓ 加窗=频谱平滑：牺牲主瓣宽度换旁瓣抑制（语音标准=汉明）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\3265143086.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [29]:
# 8 点 FFT 蝶形数字示例：奇偶分离 + 旋转因子合并（radix-2 DIT）
x = np.random.RandomState(0).randn(8)
X = np.fft.fft(x)
xe, xo = x[0::2], x[1::2]
E, O = np.fft.fft(xe), np.fft.fft(xo)
W8 = np.exp(-2j*np.pi*np.arange(4)/8)      # 旋转因子
X2 = np.concatenate([E + W8*O, E - W8*O])  # 蝶形合并：X[k]=E[k]+W^k·O[k]
print('输入 x =', np.round(x, 3).tolist())
print('两半合并 X2 =', np.round(X2, 3).tolist())
print('直接 np.fft X =', np.round(X, 3).tolist())
print('蝶形合并与直接 FFT 一致:', np.allclose(X2, X))
assert np.allclose(X2, X)
print('✓ 手撕蝶形：8 点 = 两次 4 点 + 8 次乘——这就是 O(NlogN) 的来源')

输入 x = [1.764, 0.4, 0.979, 2.241, 1.868, -0.977, 0.95, -0.151]
两半合并 X2 = [(7.073+0j), (-0.821-2.694j), (1.703+2.667j), (0.614-2.637j), (4.048+0j), (0.614+2.637j), (1.703-2.667j), (-0.821+2.694j)]
直接 np.fft X = [(7.073+0j), (-0.821-2.694j), (1.703+2.667j), (0.614-2.637j), (4.048+0j), (0.614+2.637j), (1.703-2.667j), (-0.821+2.694j)]
蝶形合并与直接 FFT 一致: True
✓ 手撕蝶形：8 点 = 两次 4 点 + 8 次乘——这就是 O(NlogN) 的来源


In [30]:
# 分帧加窗过程可视化：帧窗在波形上的位置与重叠
fs = 16000
t = np.arange(fs)/fs
x = 0.5*np.sin(2*np.pi*440*t) + 0.2*np.sin(2*np.pi*880*t)
plt.figure(figsize=(7.5, 3.2))
plt.plot(t, x, lw=0.6, alpha=0.4)
for start in (800, 1600, 2400, 3200):
    plt.axvspan(start/fs, (start+400)/fs, alpha=0.18, color='C1')
plt.xlim(0.03, 0.3); plt.xlabel('时间(s)'); plt.ylabel('幅度')
plt.title('分帧：25ms 窗(400点) 每 10ms(160点) 滑动，重叠 60%%')
plt.tight_layout(); plt.show()
print('帧长 400 点 = %.1f ms | 帧移 160 点 = %.0f ms | 重叠 %.0f%%'
      % (400/fs*1000, 160/fs*1000, 100*(400-160)/400))
print('✓ 重叠的目的：窗边缘被压低的信息在邻帧补回来（时间分辨率）')

帧长 400 点 = 25.0 ms | 帧移 160 点 = 10 ms | 重叠 60%
✓ 重叠的目的：窗边缘被压低的信息在邻帧补回来（时间分辨率）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\3748195233.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [31]:
# dB 语谱图：谐波结构一目了然（基频 + 泛音 + 噪声底）
fs = 16000
t = np.arange(int(1.0*fs))/fs
f0 = 200 + 100*t                       # 基频扫频 200→300Hz
x = np.sign(np.sin(2*np.pi*np.cumsum(f0)/fs))   # 方波（丰富谐波）
frames = np.stack([x[i:i+400]*np.hamming(400)
                  for i in range(0, len(x)-400+1, 160)])
S = np.abs(np.fft.rfft(frames, 512))
Sdb = 20*np.log10(S/S.max()+1e-9)
plt.figure(figsize=(7.5, 3.4))
plt.imshow(Sdb.T, aspect='auto', origin='lower', cmap='magma',
           extent=[0, 1.0, 0, fs/2])
plt.colorbar(label='dB'); plt.xlabel('时间(s)'); plt.ylabel('频率(Hz)')
plt.title('语谱图：横纹=谐波（基频 F0 及其整数倍），纵跳=音节边界')
plt.tight_layout(); plt.show()
print('✓ 语谱图是语音的“X 光片”：共振峰=亮带，清音=弥散噪声')

✓ 语谱图是语音的“X 光片”：共振峰=亮带，清音=弥散噪声


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\3760537675.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [32]:
# 线性谱 vs mel 谱：低频展开、高频压缩（人耳对数感知）
def mfb_quick(n_fft=512, n_mels=40, fs=16000):
    mel = lambda f: 2595*np.log10(1+f/700)
    imel = lambda m: 700*(10**(m/2595)-1)
    pts = np.linspace(mel(0), mel(fs/2), n_mels+2)
    fpts = imel(pts)
    bins = np.floor((n_fft+1)*fpts/fs).astype(int)
    W = np.zeros((n_mels, n_fft//2+1))
    for i in range(n_mels):
        for j in range(bins[i], bins[i+1]):
            W[i, j] = (j-bins[i])/(bins[i+1]-bins[i]+1e-12)
        for j in range(bins[i+1], bins[i+2]):
            W[i, j] = (bins[i+2]-j)/(bins[i+2]-bins[i+1]+1e-12)
    return W

mel_spec = mfb_quick() @ (S**2).T      # 40×帧数 mel 能量谱
plt.figure(figsize=(7.5, 3.4))
plt.subplot(1, 2, 1)
plt.imshow(np.log(S**2+1e-9).T, aspect='auto', origin='lower', cmap='viridis',
           extent=[0, 1.0, 0, fs/2])
plt.title('线性谱（0-8k Hz 等宽）', fontsize=9); plt.xlabel('时间'); plt.ylabel('Hz')
plt.subplot(1, 2, 2)
plt.imshow(np.log(mel_spec+1e-9).T, aspect='auto', origin='lower', cmap='viridis')
plt.title('mel 谱（40 维，低频密高频疏）', fontsize=9); plt.xlabel('时间'); plt.ylabel('mel 带')
plt.tight_layout(); plt.show()
print('✓ 同样信息 mel 只需 40 维 vs 线性 257 维：压缩 6 倍且更贴人耳')

✓ 同样信息 mel 只需 40 维 vs 线性 257 维：压缩 6 倍且更贴人耳


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\2702369175.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [33]:
# MFCC 系数热图：能量向低阶系数集中（DCT 的去相关+压缩）
def dct2(c, k):
    return np.dot(np.cos(np.pi/len(c)*(np.arange(len(c))+0.5)*k), c)

logmel = np.log(mel_spec+1e-9)
K = 13
mfcc = np.array([[dct2(logmel[:, t], k) for k in range(K)] for t in range(logmel.shape[1])])
plt.figure(figsize=(7.5, 3.4))
plt.imshow(mfcc.T, aspect='auto', origin='lower', cmap='coolwarm')
plt.colorbar(); plt.xlabel('帧'); plt.ylabel('MFCC 阶数 (0-12)')
plt.title('MFCC 热图：前几阶承载绝大部分能量（0 阶≈平均能量）')
plt.tight_layout(); plt.show()
print('系数能量: c0=%.2f  c1=%.2f  c2=%.2f（前 3 阶占比 %.0f%%）'
      % (np.mean(mfcc[:, 0]**2), np.mean(mfcc[:, 1]**2), np.mean(mfcc[:, 2]**2),
         100*np.sum(mfcc[:, :3]**2)/np.sum(mfcc**2)))

系数能量: c0=27868.94  c1=72.96  c2=12.88（前 3 阶占比 93%）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\3630402110.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [34]:
# CMVN 前后对比：均值/方差归一化抹平信道差异
feat = np.random.RandomState(1).randn(50, 13)*2 + 1.5
norm = (feat - feat.mean(axis=0))/feat.std(axis=0)
plt.figure(figsize=(7.5, 3.0))
plt.subplot(1, 2, 1)
plt.imshow(feat.T, aspect='auto', cmap='RdBu', vmin=-6, vmax=6)
plt.title('原始特征（均值 1.5、方差 4）', fontsize=9); plt.xlabel('帧')
plt.subplot(1, 2, 2)
plt.imshow(norm.T, aspect='auto', cmap='RdBu', vmin=-3, vmax=3)
plt.title('CMVN 后（均值 0、方差 1）', fontsize=9); plt.xlabel('帧')
plt.tight_layout(); plt.show()
print('归一到 [-2,2] 的概率 = %.1f%%（近似标准正态）' % (100*np.mean(np.abs(norm) < 2)))

C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1648959417.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(7.5, 3.0))


归一到 [-2,2] 的概率 = 96.3%（近似标准正态）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1648959417.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [35]:
# 量化阶梯可视化：输入→输出阶梯 + 锯齿误差
x = np.linspace(-1, 1, 1000)
step = 2/16                                    # 4-bit
xq = np.round(x/step)*step
plt.figure(figsize=(7.5, 3.4))
plt.plot(x, xq, lw=1.6, label='量化输出(4bit 阶梯)')
plt.plot(x, x, 'k--', lw=0.7, label='理想输入')
plt.plot(x, (xq-x)*20, label='量化误差 ×20（锯齿周期=1 LSB）', lw=0.8)
plt.xlabel('输入幅度'); plt.ylabel('输出'); plt.legend(fontsize=8)
plt.title('量化：误差均匀分布（σ≈LSB/√12 → SNR≈6.02B+1.76 dB）')
plt.tight_layout(); plt.show()
print('✓ 锯齿误差的随机性假设 = 量化噪声公式的前提')

✓ 锯齿误差的随机性假设 = 量化噪声公式的前提


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1646476998.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣8️⃣ 窗函数公式推导

矩形：$w(n)=1$，主瓣最窄但旁瓣 $-13$dB（谱泄漏严重）；

汉明：$w(n)=0.54-0.46\cos\frac{2\pi n}{M-1}$，第一旁瓣 $-43$dB（语音默认）；

汉宁：$w(n)=0.5-0.5\cos\frac{2\pi n}{M-1}$，旁瓣 $-31$dB、主瓣更宽；

加窗等价于频谱与窗谱 $W(f)$ 卷积：$X_w(f)=X(f)*W(f)$——旁瓣低 = 频谱“干净”。

DCT-II（MFCC 去相关核心）：$C_k=\sum_{n=0}^{N-1}x_n\cos\left[\frac{\pi}{N}\left(n+\tfrac12\right)k\right]$，正交、能量集中——低频系数承载大部分能量，因此 MFCC 常取 13 维。

## 🎤 信号进阶追问（8 连）

Q「为什么语谱图横纹是谐波不是基频？」→ 横纹间隔=基频 F0，纵轴位置=谐波频率；
Q「矩形窗主瓣窄为什么不好？」→ 窄主瓣分辨力好，但旁瓣泄漏会“污染”邻近频率；
Q「补零能提高频率分辨率吗？」→ 不能，只插值（分辨率=fs/N 由真实窗长决定）；
Q「MFCC 为什么取 log？」→ 幅度压缩贴近响度感知、动态范围管理；
Q「CMVN 在声纹场景能用吗？」→ 慎用——会抹掉说话人信道差异；
Q「量化误差什么时候不是白噪声？」→ 小信号/直流（误差与信号相关时）；
Q「谱减法的相位怎么办？」→ 保留原始相位（人耳对幅度敏感、相位不敏感）；
Q「8kHz 电话语音和 16kHz 区别？」→ 上限 4k vs 8k：清音高频信息（s/f）在 8k 更完整。

In [36]:
# mel 滤波器组形状：三角在 mel 域等宽 → 频率轴上低频窄、高频宽
W = mfb_quick()
plt.figure(figsize=(7.5, 3.2))
for i in range(0, 40, 5):
    plt.plot(np.linspace(0, 8000, W.shape[1]), W[i], lw=0.9)
plt.xlabel('频率 (Hz)'); plt.ylabel('滤波器增益');
plt.title('40 个三角滤波器：mel 域等间距（人耳对数感知）')
plt.tight_layout(); plt.show()
print('✓ 越往高频，三角越宽 → mel 能量谱的“非线性压缩”直观可见')

✓ 越往高频，三角越宽 → mel 能量谱的“非线性压缩”直观可见


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\839366866.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [37]:
# 全流程数字示例：正弦 → 分帧 → 加窗 → DFT → mel → MFCC（逐步可查）
fs = 16000
t = np.arange(400)/fs
x = 0.8*np.sin(2*np.pi*440*t) + 0.3*np.sin(2*np.pi*1320*t)     # 440Hz 基频+3次谐波
w = 0.54-0.46*np.cos(2*np.pi*np.arange(400)/(400-1))           # 汉明窗
fr = x*w
F = np.abs(np.fft.rfft(fr, 512))                               # 512 点 FFT
mel_e = mfb_quick() @ (F**2)                                   # 40 维 mel 能量
c0 = dct2(np.log(mel_e+1e-9), 0)
print('① 帧能量 = %.4f（分帧加窗后）' % np.sum(fr**2))
print('② DFT 峰值 bin = %.0f Hz（= 基频 440Hz ✓）' % (np.argmax(F)/512*fs))
print('③ 二次峰 bin = %.0f Hz（= 1320Hz ✓）' % (np.argsort(F)[-2]/512*fs))
print('④ mel 能量最大值在带 %d（≈440Hz 所在三角）' % np.argmax(mel_e))
print('⑤ MFCC c0 = %.3f（= 40 维 log-mel 均值 × 权重，能量代表）' % c0)
assert abs(np.argmax(F)/512*fs - 440) < 20
print('✓ 全链路示例：每步的中间量都打印验证，特征提取不再黑盒')

① 帧能量 = 57.8777（分帧加窗后）
② DFT 峰值 bin = 438 Hz（= 基频 440Hz ✓）
③ 二次峰 bin = 469 Hz（= 1320Hz ✓）
④ mel 能量最大值在带 7（≈440Hz 所在三角）
⑤ MFCC c0 = -101.706（= 40 维 log-mel 均值 × 权重，能量代表）
✓ 全链路示例：每步的中间量都打印验证，特征提取不再黑盒


In [38]:
# ASR 全流程框图：波形 → 特征 → 声学模型 → 解码 → 文本（数据流一目了然）
fig, ax = plt.subplots(figsize=(9.5, 3.4))
steps = ['波形\n16k 采样', '分帧加窗\n25ms/10ms', 'FFT/STFT\n语谱图', 'mel 滤波\n对数感知', 'log+CMVN\nFbank/MFCC', '声学模型\nCTC/AED', '语言模型\n+解码', '文本输出']
x0, w, gap, y, h = 0.02, 0.118, 0.006, 0.50, 0.34
for i, s in enumerate(steps):
    x = x0 + i*(w+gap)
    ax.add_patch(plt.Rectangle((x, y), w, h, fc='#dbe9ff', ec='#1f4e79', lw=1.2))
    ax.text(x+w/2, y+h/2, s, ha='center', va='center', fontsize=8)
    if i < len(steps)-1:
        ax.annotate('', xy=(x+w+gap-0.001, y+h/2), xytext=(x+w+0.001, y+h/2),
                    arrowprops=dict(arrowstyle='->', lw=1.2, color='#1f4e79'))
ax.text(0.5, 0.10, '训练侧：波形+标注 → 损失回传；推理侧：只跑正向 → 实时性取决于声学+解码',
        ha='center', fontsize=9, color='#555555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('语音识别全流程：每个模块独立可替换（这就是“模块化”工程架构）', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 8 大模块：特征(2-5) 决定输入质量，声学/语言(6-7) 决定准确率，延迟主要在解码')

✓ 8 大模块：特征(2-5) 决定输入质量，声学/语言(6-7) 决定准确率，延迟主要在解码


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\762115394.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [39]:
# 特征提取细化框图：每步“输入→处理→输出”三要素
fig, ax = plt.subplots(figsize=(9, 3.2))
stages = [('原始波形', '预加重\ny[n]=x[n]-ax[n-1]', '平坦频谱'),
          ('一帧 400 点', '加窗（汉明）\n旁瓣 -43dB', '消除泄漏'),
          ('加窗信号', 'FFT 512 点\n256 频点', '幅度谱 X[k]'),
          ('|X[k]|²', 'mel 滤波组\n26~40 三角', 'mel 能量'),
          ('log-mel', 'DCT 取前 13\n去相关', 'MFCC 向量')]
x0, w, gap, y, h = 0.03, 0.19, 0.005, 0.45, 0.36
for i, (inp, op, out) in enumerate(stages):
    x = x0 + i*(w+gap)
    ax.add_patch(plt.Rectangle((x, y), w, h, fc='#e2efda', ec='#376027', lw=1.1))
    ax.text(x+w/2, y+h-0.07, inp, ha='center', fontsize=8, color='#333')
    ax.text(x+w/2, y+h/2-0.02, op, ha='center', fontsize=7.5, color='#1f4e79')
    ax.text(x+w/2, y+0.04, '→ ' + out, ha='center', fontsize=8, color='#555')
    if i < len(stages)-1:
        ax.annotate('', xy=(x+w+gap-0.001, y+h/2), xytext=(x+w+0.001, y+h/2),
                    arrowprops=dict(arrowstyle='->', lw=1.1, color='#376027'))
ax.text(0.5, 0.08, '记忆点：每步都在“去冗余/贴感知/规范化” —— 特征工程的本质', ha='center', fontsize=9, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('特征提取 5 步流水线：输入/操作/输出三要素', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 面试答法：先背流程图，再按图逐个展开公式与数值')

✓ 面试答法：先背流程图，再按图逐个展开公式与数值


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1880555559.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🧮 特征域核心公式推导（面试默写版）

**① DFT/FFT**：$X_k=\sum_{n=0}^{N-1}x_n e^{-j2\pi kn/N}$，频率分辨率 $\Delta f=f_s/N$；只取前 $N/2+1$ 个（实信号共轭对称）。

**② 预加重**：$y_n=x_n-a\,x_{n-1}$（$a\approx0.97$），提升高频——嘴唇辐射效应使高频自然衰减约 6 dB/倍频程。

**③ mel 刻度**：$m=2595\log_{10}(1+f/700)$；反变换 $f=700(10^{m/2595}-1)$。滤波器组中心频率在 mel 域等距 → 频率轴上低频密高频疏。

**④ log**：$E_m=\log\left(\sum_k |X_k|^2 W_m(k)\right)$，对数压缩模拟响度感知，并使分布接近高斯。

**⑤ DCT**：$C_k=\sum_{m=0}^{M-1}\log E_m\cos\!\left[\tfrac{\pi}{M}(m+\tfrac12)k\right]$，取 $k=0,\dots,12$；C0 反映整体能量，C1-C12 描述频谱形状。

**⑥ CMVN**：$\hat x=\dfrac{x-\mu_{\text{句}}}{\sigma_{\text{句}}}$，按句子减均值除方差，抗说话人/信道/麦克风差异。

> 推导思路：DFT 正交基 → mel 感知刻度 → 对数域 → 去相关 —— 每步都有“为什么”。

In [40]:
# 混叠数字示例：9kHz 被 8k 采样 → 折叠成 1kHz（看不见 9k！）
fs8 = 8000
t8 = np.arange(400)/fs8
y8 = np.sin(2*np.pi*9000*t8)                       # 高于 fs/2=4k 的分量
peaks = np.where(np.diff(np.sign(np.diff(y8))) < 0)[0]
f_est = fs8/np.mean(np.diff(peaks)) if len(peaks) > 1 else 0.0
f_alias = abs(9000 % fs8)
if f_alias > fs8/2: f_alias = fs8 - f_alias
print('采样后可见频率 ≈ %.0f Hz（理论混叠 %.0f Hz，真实 9000 Hz）' % (f_est, f_alias))
plt.figure(figsize=(7.5, 2.6))
plt.plot(t8, y8, lw=0.8)
plt.title('混叠：9kHz 经 8k 采样后看起来是 1kHz（奈奎斯特折叠）', fontsize=10)
plt.xlabel('时间 (s)'); plt.ylabel('幅度'); plt.tight_layout(); plt.show()
assert abs(f_est - 1000) < 60
print('✓ 高于 fs/2 的分量被折叠回 0~fs/2 → 采样前必须抗混叠低通滤波')

采样后可见频率 ≈ 1000 Hz（理论混叠 1000 Hz，真实 9000 Hz）
✓ 高于 fs/2 的分量被折叠回 0~fs/2 → 采样前必须抗混叠低通滤波


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1831016702.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel('时间 (s)'); plt.ylabel('幅度'); plt.tight_layout(); plt.show()


In [41]:
# 窗函数频谱对比：主瓣宽度(时间分辨率) vs 旁瓣抑制(泄漏)
N = 200
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
for name, w in {'矩形': np.ones(N), '汉明': np.hamming(N),
                '汉宁': np.hanning(N), '布莱克曼': np.blackman(N)}.items():
    ax[0].plot(w, lw=1.2, label=name)
    H = np.abs(np.fft.rfft(w*1000, 4096))
    Hdb = 20*np.log10(H/H.max()+1e-12)
    ax[1].plot(np.linspace(0, 0.5, len(Hdb)), Hdb, lw=1.1, label=name)
ax[0].set_title('时域窗形状'); ax[0].legend(fontsize=8); ax[0].set_xlabel('n')
ax[1].set_title('频谱（主瓣/旁瓣，dB）'); ax[1].legend(fontsize=8)
ax[1].set_xlabel('归一化频率'); ax[1].set_ylim(-120, 5); ax[1].set_ylabel('dB')
plt.tight_layout(); plt.show()
print('✓ 矩形旁瓣≈-13dB（最差），汉明≈-43dB：泄漏下降 ~30dB，主瓣略宽')

✓ 矩形旁瓣≈-13dB（最差），汉明≈-43dB：泄漏下降 ~30dB，主瓣略宽


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\3824447900.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [42]:
# 自相关基频估计数字示例：峰所在滞后 = 1 个基频周期
fs = 16000
t = np.arange(400)/fs
x = 0.8*np.sin(2*np.pi*233*t) + 0.2*np.random.RandomState(4).randn(400)
xc = x - x.mean()
ac = np.correlate(xc, xc, 'full')[399:]
lags = np.arange(40, 250)                 # 只搜 64~400 Hz（跳过零滞后附近的伪峰）
peak = lags[np.argmax(ac[lags])]
f0 = fs/peak
print('自相关峰滞后 = %d 样本 → F0 ≈ %.1f Hz（真实 233 Hz）' % (peak, f0))
plt.figure(figsize=(7.5, 2.6))
plt.plot(lags, ac[lags], lw=1.0)
plt.axvline(peak, color='r', ls='--', lw=1)
plt.xlabel('滞后（样本）'); plt.ylabel('自相关'); plt.title('自相关：峰在 1/F0（浊音周期性）', fontsize=10)
plt.tight_layout(); plt.show()
assert abs(f0 - 233) < 8
print('✓ 清音无周期 → 自相关无峰；浊音峰 = 基频周期 → 判浊/清音 + 估 F0')

自相关峰滞后 = 67 样本 → F0 ≈ 238.8 Hz（真实 233 Hz）
✓ 清音无周期 → 自相关无峰；浊音峰 = 基频周期 → 判浊/清音 + 估 F0


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\2519746108.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 📜 经典算法伪代码（背下来就能手撕）

**能量 VAD（最简可用版）**：
```
对每一帧：
    E_t = mean(x[i:i+400]^2)                    # 短时能量
    v_t = 1 if E_t > 0.5 * mean(E) else 0       # 阈值=全局能量均值的一半
平滑：去毛刺（<3 帧的脉冲删掉；<gap 的间隙填上）
输出：语音段 (start, end) 列表
```

**预加重 + 分帧 + 加窗（10 行内实现）**：
```
y = [x[0]] + [x[n]-0.97*x[n-1] for n in 1..]   # 预加重
frames = [y[i:i+400]*hamming(400) for i in 0,160,...]  # 25ms/10ms 窗
```

**MFCC 提取（一句话流程）**：预加重→分帧加窗→|FFT|²→mel 滤波→log→DCT(13)→(±Δ)。

> 面试常考：把上述任一步的复杂度说清楚（FFT O(N logN)，mel 滤波 O(M·N/2)）。

## 🎤 特征工程追问（8 连）

Q「为什么用 dB 画语谱图？」→ 动态范围 ~120dB，线性图亮部把暗部全吞了；
Q「mel 滤波为什么三角不是矩形？」→ 三角窗=软边界，避免硬截断频谱泄漏；
Q「40 维 mel vs 26 维哪个常用？」→ 26（电话）~40（宽带），维度=频率分辨率与计算量折中；
Q「MFCC 要不要 CMVN？」→ 传统 GMM 要（高斯假设）；DNN 时代 fbank+CMVN 是标配；
Q「相位去哪了？」→ 特征只用幅度：人耳对幅度敏感、相位对识别贡献低（合成才在意相位）；
Q「采样率不够会怎样？」→ 混叠 → 特征错误 → 识别率崩；工程上先查 fs 与带宽假设；
Q「加窗重叠多少合适？」→ 50%~75%：重叠越多帧间越平滑，但计算量线性增加；
Q「特征该不该归一到同一量纲？」→ 每维减均值除方差（CMVN/标准化）是 DNN 输入默认动作。

## ✅ 自测（10 题，先答后对答案）

1. 16k 采样，帧长 25ms、帧移 10ms：一帧多少点？每秒多少帧？
2. 400 点加窗做 512 点 FFT：频点数？频率分辨率？（$16000/512=31.25$Hz）
3. 为什么先预加重再分帧？（补偿嘴唇辐射，防高频信息被淹没）
4. 汉明窗主瓣宽还是矩形窗宽？为什么还选它？（旁瓣 -43dB 换低泄漏）
5. mel 刻度 1000Hz 对应多少 mel？（≈1000·ln 单位：$2595\log_{10}(1+1000/700)\approx1000$）
6. 语谱图横/纵轴含义？（时间/频率，颜色=能量 dB）
7. MFCC 13 维里 C0 代表什么？（整体能量）
8. CMVN 在什么阶段做、为什么？（特征后、模型前：抗信道/说话人差异）
9. 混叠能事后修复吗？（不能——信息已丢失；只能事前抗混叠 LPF）
10. 端到端系统还要这些特征吗？（可不要：直接学原始波形/学习特征，但工程仍常用）

In [43]:
# 全链路数字示例 2：任意语音帧 → 13 维 MFCC（与示例1同链路，不同输入）
fs = 16000
t = np.arange(320)/fs
x = 0.5*np.sin(2*np.pi*500*t) + 0.25*np.sin(2*np.pi*1500*t)
fr = x*np.hamming(320)
F = np.abs(np.fft.rfft(fr, 512))
mel_e = mfb_quick(n_fft=512, n_mels=26, fs=16000) @ (F**2)
lm = np.log(mel_e+1e-9)
mfcc13 = np.array([dct2(lm, k) for k in range(13)])
print('帧能量 = %.5f | mel 带前 3 能量 = %s' % (np.sum(fr**2), np.round(mel_e[:3], 4)))
print('MFCC c0..c2 = %s（c0=能量，c1/c2=频谱斜率与形状）' % np.round(mfcc13[:3], 4))
assert len(mfcc13) == 13 and np.isfinite(mfcc13).all()
print('✓ 结论：任意帧都能 320→13 维压缩 25 倍，且保留区分性信息（下游可训练）')

帧能量 = 19.80890 | mel 带前 3 能量 = [0.2036 0.293  0.3753]
MFCC c0..c2 = [-57.8587  53.2717 -13.513 ]（c0=能量，c1/c2=频谱斜率与形状）
✓ 结论：任意帧都能 320→13 维压缩 25 倍，且保留区分性信息（下游可训练）


## 🍵 白话开篇：语音处理到底在干嘛？

想象你录了一段话。电脑拿到的只是一串数字：`x[0], x[1], x[2], ...`，每 1/16000 秒一个数（16k 采样）。
这一串数字里**藏着**：说了什么（内容）、谁在说（音色）、多快多慢（节奏）、多大声（音量）。
可机器不会像人一样“听”，它只会算——所以我们得先把声音变成**机器好算的表格**（特征）。

**本节的逻辑链（背下来）**：
```
波形(一串数) → 分帧(切成小段) → 加窗(防泄漏) → FFT(看频率成分)
→ |X|²(能量) → mel 滤波(按人耳刻度加权) → log(贴近听觉响度)
→ DCT 取前 13(压缩+去相关) = MFCC 向量（每 10ms 一个）
```

一句话记忆：**波形讲“时间”，频谱讲“频率”，特征把两者压成每帧一个短向量**。
后面所有语音算法（GMM/HMM/CTC/Transformer）吃的都是这种向量。

## 🍵 白话讲透：为什么必须“分帧 + 加窗”？

**为什么不能整段一次 FFT？** 因为 FFT 假设信号是**平稳**的（统计性质不变），
而语音 200ms 内就在变（元音→辅音→元音）。整段 FFT 只给你“整段平均”的频谱，**时间信息全丢**——
你不知道“b”在第 3 帧还是第 8 帧。

**为什么分帧（25ms/10ms）？** 类比电影：1 秒画面拆成 24 帧，帧内动作近似不变。
25ms 内声带和声道基本稳定 → 每帧可以做 FFT → 得到“这 25ms 有哪些频率”。
帧与帧重叠 10ms → 相邻帧平滑过渡，信息不丢。

**为什么还要加窗？** 因为 FFT 默认这段信号是“周期重复”的——可 25ms 的片段首尾不连续，
硬接起来会“咔哒”一声，频谱上表现为**旁瓣泄漏**（明明只有 440Hz，却到处都漏出能量）。
汉明窗把两端压到 0，首尾平滑接上 → 泄漏大降（-43dB），代价是主瓣变宽一点点（频率分辨率略降）。

**一句话**：分帧管“时间切片”，加窗管“切片边缘别漏能量”。两者缺一，特征就脏。

## 🔍 公式拆解 ①：DFT 到底在算什么？

$$X_k=\sum_{n=0}^{N-1} x_n\, e^{-j2\pi kn/N}$$

**逐符号解释**：
- $x_n$：第 $n$ 个采样点（一帧里的第 n 个数）；
- $N$：一帧的点数（如 400）；
- $k$：频率序号（$k=0,\dots,N/2$，对应频率 $f_k=k\cdot f_s/N$）；
- $e^{-j2\pi kn/N}$：一个**旋转的探针**（复指数），频率就是 $k$；
- $X_k$：信号与“频率为 k 的探针”的**匹配程度**（点积）。

**直觉（一句话）**：把信号和“每秒转 k 圈的指针”做点积——信号里真有这个频率，点积就大。
$|X_k|$ 大 = 这个频率成分强；$|X_k|\approx0$ = 没有这个频率。

**为什么只取前 $N/2+1$ 个？** 实信号频谱**共轭对称**（后半部分是前半的镜像），信息不重复。

**手算 1 例（N=4）**：$x=[1,0,-1,0]$，
$X_1=1\cdot e^0+0\cdot e^{-j\pi/2}+(-1)\cdot e^{-j\pi}+0\cdot e^{-j3\pi/2}=1+0+1+0=2$（实部），
说明这个信号在 $k=1$（$f=f_s/4$）处有能量——你甚至能手工验一遍。

## 🔍 公式拆解 ②：加窗与汉明窗

$$y_n = x_n \cdot w_n,\qquad w_n^{\text{Hamm}} = 0.54 - 0.46\cos\left(\frac{2\pi n}{N-1}\right)$$

**逐符号解释**：
- $x_n$：原始采样点；$w_n$：窗函数第 n 个系数；$y_n$：加窗后的点；
- $n=0,\dots,N-1$：窗内序号；
- 汉明窗：$n=0$ 时 $\cos(0)=1$ → $w_0=0.54-0.46=0.08$（几乎压到 0）；
  $n=(N-1)/2$ 时 $\cos(\pi)=-1$ → $w=0.54+0.46=1.0$（中间全开）；
- 为什么两个常数？0.54/0.46 恰好让旁瓣互相抵消到 -43dB（工程调出来的最优）。

**直觉**：窗 = 一个“软幕布”，中间透明、两边渐暗，把一帧裁成“中间实、两端虚”的小段。

**其他窗**：矩形（$w_n=1$，主瓣最窄但旁瓣 -13dB 最差）；汉宁（0.5-0.5cos，旁瓣 -31dB）；
布莱克曼（旁瓣 -58dB，主瓣更宽）。**规则**：想分辨近频选窄主瓣；想压制杂散选低旁瓣。

## 🔍 公式拆解 ③：mel 刻度——人耳的“变焦镜头”

$$m = 2595\,\log_{10}\left(1+\frac{f}{700}\right)$$

**逐符号解释**：
- $f$：物理频率（Hz）；$m$：感知频率（mel）；
- $700$：参考常数——大约 700Hz 以下，人耳近似线性感知；
- $2595$：比例常数——1kHz 时正好 $\approx 1000$ mel（把尺度校准到“好记”）；
- $\log_{10}$：对数——**高频被压缩**（8kHz→16kHz 在 mel 上只占一小段）。

**直觉**：人耳像相机的**变焦镜头**——低频区域对频率变化敏感（1Hz 差都听得见），
高频区域迟钝（10kHz 和 11kHz 几乎一样）。物理上等距的频率，感知上不是等距的。
所以 mel 滤波组在 mel 轴上等距放三角窗 → 物理轴上低频密、高频疏。

**反变换**：$f=700(10^{m/2595}-1)$——做题时两个方向都要会。

**数字验证**：$f=1000$ → $m=2595\log_{10}(1+1000/700)=2595\log_{10}(2.4286)\approx 1000$ ✓；
$f=4000$ → $m\approx 2146$（只比 1k 涨了 1 倍多，而频率涨了 4 倍）。

## 🔍 公式拆解 ④：CMVN——给特征“校准”

$$\hat x_t = \frac{x_t - \mu}{\sigma},\qquad \mu=\frac1T\sum_{t=1}^{T}x_t,\quad \sigma^2=\frac1T\sum_{t=1}^{T}(x_t-\mu)^2$$

**逐符号解释**：
- $x_t$：第 t 帧的特征向量（如 26 维 log-mel）；$T$：本句帧数；
- $\mu$：句子均值（每维一个数）——代表“这个人整体说话的水平”；
- $\sigma$：句子标准差——代表“起伏幅度”；
- 分子减均值：**去掉说话人/话筒的整体偏移**（有人嗓门大、有人话筒闷）；
- 分母除标准差：**把起伏拉齐**（有人平铺直叙、有人抑扬顿挫）。

**直觉（比喻）**：两个同学考试，一个年级 80 分、一个年级 60 分——直接比分数不公平；
CMVN = 各自减去年级平均、再除以年级标准差，**比“相对水平”**。
识别也一样：同一句话不同人/不同设备录出来，特征分布不同 → 模型要先“去掉这人这设备的底色”。

**工程要点**：训练时按句统计（或在线滑动统计）；必须在**模型输入之前**做，
且测试与训练用**同一套统计方式**，否则分布漂移。

In [44]:
# 分帧+加窗过程可视化：看清“切段-开窗-送 FFT”
fs = 16000
t = np.arange(int(0.2*fs))/fs
x = 0.7*np.sin(2*np.pi*440*t) + 0.3*np.sin(2*np.pi*1100*t)
win = np.hamming(400)
fig, ax = plt.subplots(3, 1, figsize=(8.5, 5.0), sharex=True)
ax[0].plot(t*1000, x, lw=0.7); ax[0].set_title('① 原始波形（0.2s）', fontsize=9)
for i in range(0, 6):
    ax[1].plot(np.arange(400)/fs*1000, x[i*160:i*160+400], lw=0.6, alpha=0.8)
ax[1].set_title('② 分帧（25ms 帧 / 10ms 帧移 → 相邻帧重叠 60%）', fontsize=9)
for i in range(0, 6):
    ax[2].plot(np.arange(400)/fs*1000, x[i*160:i*160+400]*win, lw=0.6, alpha=0.8)
ax[2].set_title('③ 加窗（汉明）：两端压到 0，消除频谱泄漏', fontsize=9)
for a in ax: a.set_xlabel('时间 (ms)', fontsize=8)
plt.tight_layout(); plt.show()
print('✓ 三步走：每帧 400 点 → 与窗逐点相乘 → 就能送 FFT（得 257 个频点）')

✓ 三步走：每帧 400 点 → 与窗逐点相乘 → 就能送 FFT（得 257 个频点）


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\1820318061.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [45]:
# CMVN 数字示例：两段不同“音量/偏移”的语音，标准化后分布对齐
rng = np.random.RandomState(0)
t1 = np.arange(500)/1000
sigA = 0.05 + 0.3*np.sin(2*np.pi*5*t1) + 0.01*rng.randn(500)   # 小而平稳
sigB = 0.5 + 0.9*np.sin(2*np.pi*5*t1) + 0.15*rng.randn(500)    # 大而起伏
def cmvn(s):
    mu, sd = s.mean(), s.std()
    return (s - mu)/sd
print('原始: A 均值 %.3f 标准差 %.3f | B 均值 %.3f 标准差 %.3f（差很大）'
      % (sigA.mean(), sigA.std(), sigB.mean(), sigB.std()))
print('CMVN: A 均值 %.2e 标准差 %.3f | B 均值 %.2e 标准差 %.3f（几乎一样）'
      % (cmvn(sigA).mean(), cmvn(sigA).std(), cmvn(sigB).mean(), cmvn(sigB).std()))
assert abs(cmvn(sigA).std() - 1) < 1e-9 and abs(cmvn(sigA).mean()) < 1e-9
print('✓ 减均值除标准差后：均值→0、标准差→1 → 模型只看“形状”不看“底色”')

原始: A 均值 0.088 标准差 0.209 | B 均值 0.605 标准差 0.649（差很大）
CMVN: A 均值 1.42e-17 标准差 1.000 | B 均值 -8.53e-17 标准差 1.000（几乎一样）
✓ 减均值除标准差后：均值→0、标准差→1 → 模型只看“形状”不看“底色”


In [46]:
# mel 滤波器组可视化：物理轴上低频密、高频疏（人耳变焦镜头）
W = mfb_quick(n_fft=512, n_mels=13, fs=16000)
freqs = np.linspace(0, 8000, W.shape[1])
plt.figure(figsize=(8.5, 3.4))
for i in range(13):
    plt.plot(freqs, W[i], lw=1.0)
plt.xlabel('频率 (Hz)'); plt.ylabel('权重')
plt.title('13 个三角滤波器：低频区窄而密、高频区宽而疏（mel 刻度）', fontsize=10)
plt.tight_layout(); plt.show()
print('✓ 每个三角窗 = 对一段频率“加权求和” → 一个 mel 能量；13 窗 → 13 维 mel 向量')

✓ 每个三角窗 = 对一段频率“加权求和” → 一个 mel 能量；13 窗 → 13 维 mel 向量


C:\Users\24260\AppData\Local\Temp\ipykernel_20284\2017463764.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## ⚠️ 本章高频踩坑（面试也爱问）

1. **采样率记错**：16k 采样 → 最高可表示 8kHz；说“8k 采样”范围是 4kHz；电话即 8k。
2. **帧移/帧长反了**：帧长 25ms=400 点（16k），帧移 10ms=160 点；不要写成帧长 10ms。
3. **FFT 点数**：400 点输入可做 512 点 FFT（补零），频点=512/2+1=257；分辨率=16000/512=31.25Hz。
4. **加窗顺序**：必须先分帧再逐帧加窗；对整段波形加窗会破坏帧间连续性。
5. **mel 与 MFCC 混为一谈**：mel=滤波+log（26~40 维，DNN 时代够用）；MFCC=再 DCT 取 13（传统 GMM 时代）。
6. **CMVN 时机**：一定在特征后、模型前；测试与训练统计口径必须一致。
7. **相位丢弃**：特征只用幅度；丢相位 = 丢声源定位信息，但对内容识别无害（合成才要相位）。
8. **忘记 Δ/ΔΔ**：传统系统常拼一阶/二阶差分（每帧 13→39 维）捕捉动态。

> 自查：把上面 8 条都能讲出一句“为什么”，本节就过关了。

## 🧭 特征家族对比：什么时候用哪个？

| 特征 | 维度 | 优点 | 缺点 | 典型场景 |
|---|---|---|---|---|
| 波形 (raw) | 全采样 | 信息无损 | 维度爆炸、难学 | 端到端（少量） |
| 幅度谱 \|X\| | N/2+1 | 直观 | 冗余大、不贴听觉 | 频谱分析 |
| log-mel (Fbank) | 26~80 | 贴听觉、DNN 友好 | 相关性强 | **主流前端** |
| MFCC | 13~39(+Δ) | 去相关、紧凑 | 信息有损 | 传统 GMM-HMM |
| 谱图图像 | 高 | 可当图像做 CNN | 计算重 | 视觉前端（少见） |

**一句话决策**：新项目默认 log-mel + CMVN；只有做传统 GMM 或强实时压缩时才回头用 MFCC。

**前沿补充**：wav2vec/HuBERT 直接把波形/隐向量当“特征”喂 Transformer——
但**工程产品里 log-mel 依旧主力**（简单、稳定、省算力），这也是面试常考点。

## 🏁 本章学完清单（对照自查）

- [ ] 能解释：为什么采样、采样率上限（奈奎斯特）；
- [ ] 能解释：为什么分帧、为什么加窗、窗的选择权衡；
- [ ] 能手写：DFT 公式并说明每个符号；
- [ ] 能解释：mel 刻度为什么对数、mel→MFCC 为什么 DCT；
- [ ] 能算：给定 fs/N 算分辨率与频点数；
- [ ] 能画：分帧→加窗→FFT→mel→log→DCT 全流程图；
- [ ] 能答：log-mel vs MFCC vs raw 波形怎么选；
- [ ] 能做：用 numpy 从波形提取 13 维 MFCC（本 notebook 两个示例）。

> 下一站：拿到特征后，怎么用 GMM-HMM 建模——见下一本 notebook。